# 🔧 Fine-Tuning — Hands-On Lab## Prepare data, generate synthetic examples, fine-tune via API, compare approaches.**What this notebook does:**- Prepare training data in ALL 3 formats (Alpaca, ShareGPT, ChatML)- Validate and clean your dataset (dedup, quality checks)- Generate synthetic training data with GPT-4 (Self-Instruct + Evol-Instruct)- Fine-tune a model via OpenAI API (no GPU needed!)- Prepare DPO preference pairs for alignment- LoRA/QLoRA config walkthrough (GPU code you can paste into Colab)- Cost calculator: API fine-tuning vs self-hosted vs no fine-tuning**Setup:**```pip install openai datasets```**Cost:** ~$0.50 for synthetic data generation. Fine-tuning costs depend on dataset size (~$1-5 for small demos).**GPU note:** Programs 1-5 run on ANY laptop (no GPU). Program 6 shows GPU code you can paste into Google Colab (free T4 GPU).

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# pip install openai datasets
from openai import OpenAI
import json, hashlib, random, os, time
from collections import Counter

client = OpenAI()

def ask(prompt, model="gpt-4o-mini", temperature=0, system=None, json_mode=False):
    messages = []
    if system: messages.append({"role":"system","content":system})
    messages.append({"role":"user","content":prompt})
    kwargs = {"model":model, "messages":messages, "temperature":temperature}
    if json_mode:
        kwargs["response_format"] = {"type":"json_object"}
    r = client.chat.completions.create(**kwargs)
    return r.choices[0].message.content.strip(), r.usage.total_tokens

answer, _ = ask("Say 'Fine-tuning ready' in 3 words.")
print(f"Setup complete! {answer}")

---# 1️⃣ Training Data Formats (The 3 Standards)**Analogy:** Instruction tuning is like showing a new employee 1000 examples of "customer asked X, you should respond Y." After seeing enough examples, they learn the pattern.**Three formats:** Alpaca (homework worksheets), ShareGPT (chat conversations), ChatML (OpenAI standard). We build all 3.

In [ ]:
# ============================================================
# 3 DATA FORMATS: Every fine-tuning framework expects one of these
# We'll create the SAME examples in all 3 formats
# ============================================================

# ---------- RAW EXAMPLES (what we know) ----------

raw_examples = [
    {
        "task": "Classify the sentiment of this review",
        "input": "This laptop is absolutely amazing! Best purchase I've made all year.",
        "good_response": "POSITIVE"
    },
    {
        "task": "Extract the product and issue",
        "input": "My AirPods keep disconnecting from my iPhone every 10 minutes",
        "good_response": '{"product": "AirPods", "issue": "disconnecting", "frequency": "every 10 minutes"}'
    },
    {
        "task": "Summarize this customer complaint in one sentence",
        "input": "I ordered a blue sweater size medium but received a red sweater size large. I called support three times and each time was told it would be resolved but nothing happened. It has been 2 weeks.",
        "good_response": "Customer received wrong color and size, and three support calls over two weeks have not resolved the issue."
    },
    {
        "task": "Classify the support ticket priority",
        "input": "Our entire production database is down and no customers can access the platform",
        "good_response": "CRITICAL"
    },
    {
        "task": "Write a polite response to this angry customer",
        "input": "This is the WORST service I have ever experienced! I want a full refund NOW!",
        "good_response": "I completely understand your frustration, and I sincerely apologize for your experience. Let me look into your account right away and process that refund for you. Could you please share your order number?"
    },
]

print(f"We have {len(raw_examples)} raw examples. Now converting to 3 formats:\n")


# ---------- FORMAT 1: ALPACA (like a homework worksheet) ----------
# Used by: Axolotl, many open-source tools

alpaca_data = []
for ex in raw_examples:
    alpaca_data.append({
        "instruction": ex["task"],
        "input": ex["input"],
        "output": ex["good_response"]
    })

print("FORMAT 1 — ALPACA (homework worksheet):")
print(json.dumps(alpaca_data[0], indent=2))
print()


# ---------- FORMAT 2: SHAREGPT (chat conversation) ----------
# Used by: Axolotl (sharegpt format), many Asian-origin models

sharegpt_data = []
for ex in raw_examples:
    sharegpt_data.append({
        "conversations": [
            {"from": "human", "value": f"{ex['task']}\n\n{ex['input']}"},
            {"from": "gpt", "value": ex["good_response"]}
        ]
    })

print("FORMAT 2 — SHAREGPT (chat conversation):")
print(json.dumps(sharegpt_data[0], indent=2))
print()


# ---------- FORMAT 3: CHATML / OPENAI (the API standard) ----------
# Used by: OpenAI fine-tuning, most production systems

chatml_data = []
for ex in raw_examples:
    chatml_data.append({
        "messages": [
            {"role": "system", "content": "You are a helpful customer support classifier."},
            {"role": "user", "content": f"{ex['task']}\n\n{ex['input']}"},
            {"role": "assistant", "content": ex["good_response"]}
        ]
    })

print("FORMAT 3 — CHATML / OpenAI (API standard):")
print(json.dumps(chatml_data[0], indent=2))


# ---------- SAVE ALL 3 ----------

for name, data in [("alpaca", alpaca_data), ("sharegpt", sharegpt_data), ("chatml", chatml_data)]:
    filename = f"training_data_{name}.jsonl"
    with open(filename, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    print(f"\n  Saved: {filename} ({len(data)} examples)")

print("\n💡 Alpaca: simple, single-turn. Good for classification/extraction tasks.")
print("💡 ShareGPT: multi-turn conversations. Good for chatbot fine-tuning.")
print("💡 ChatML: OpenAI API standard. Use this for OpenAI fine-tuning.")

---# 2️⃣ Data Quality Checks (Garbage In = Garbage Out)**Analogy:** Training a doctor. Would you rather they study 1000 carefully reviewed cases from expert doctors, or 100,000 random cases from the internet? Quality > Quantity. The Stanford LIMA paper proved: 1000 curated examples matched 52,000 auto-generated ones.**What we check:** Duplicates, empty fields, balance, length outliers, format consistency.

In [ ]:
# ============================================================
# DATA QUALITY: Check your training data BEFORE fine-tuning
# Finding problems BEFORE training saves days of debugging
# ============================================================

def check_data_quality(data, format_type="chatml"):
    """
    Run 6 quality checks on your training data.
    Returns a report with issues found.
    """
    
    issues = []
    stats = {
        "total": len(data),
        "checks_passed": 0,
        "checks_failed": 0,
    }
    
    print(f"  Checking {len(data)} examples...\n")
    
    # CHECK 1: Empty or missing fields
    empty_count = 0
    for i, item in enumerate(data):
        if format_type == "chatml":
            for msg in item.get("messages", []):
                if not msg.get("content", "").strip():
                    empty_count += 1
                    issues.append(f"  Row {i}: empty content in {msg.get('role')} message")
    
    status = "PASS" if empty_count == 0 else "FAIL"
    print(f"  {'✅' if status=='PASS' else '❌'} Check 1 — Empty fields: {empty_count} found")
    stats["checks_passed" if status=="PASS" else "checks_failed"] += 1
    
    # CHECK 2: Duplicates (same input = wasted training)
    seen_hashes = set()
    dupe_count = 0
    for item in data:
        content = json.dumps(item, sort_keys=True)
        h = hashlib.md5(content.encode()).hexdigest()
        if h in seen_hashes:
            dupe_count += 1
        seen_hashes.add(h)
    
    status = "PASS" if dupe_count == 0 else "FAIL"
    print(f"  {'✅' if status=='PASS' else '❌'} Check 2 — Duplicates: {dupe_count} found")
    stats["checks_passed" if status=="PASS" else "checks_failed"] += 1
    
    # CHECK 3: Response length distribution
    lengths = []
    for item in data:
        if format_type == "chatml":
            assistant_msgs = [m for m in item["messages"] if m["role"] == "assistant"]
            for m in assistant_msgs:
                lengths.append(len(m["content"].split()))
    
    avg_len = sum(lengths) / max(len(lengths), 1)
    min_len = min(lengths) if lengths else 0
    max_len = max(lengths) if lengths else 0
    
    short_count = sum(1 for l in lengths if l < 3)
    status = "PASS" if short_count == 0 else "WARN"
    print(f"  {'✅' if status=='PASS' else '⚠️'} Check 3 — Response lengths: min={min_len}, avg={avg_len:.0f}, max={max_len} words")
    if short_count:
        print(f"           {short_count} responses under 3 words (might be too short)")
    
    # CHECK 4: Task type balance
    if format_type == "chatml":
        first_words = []
        for item in data:
            user_msgs = [m for m in item["messages"] if m["role"] == "user"]
            if user_msgs:
                first_word = user_msgs[0]["content"].split()[0].lower()
                first_words.append(first_word)
        
        distribution = Counter(first_words).most_common(5)
        dominant_pct = distribution[0][1] / len(data) * 100 if distribution else 0
        
        status = "PASS" if dominant_pct < 60 else "WARN"
        print(f"  {'✅' if status=='PASS' else '⚠️'} Check 4 — Task balance: top task is {dominant_pct:.0f}% of data")
        for word, count in distribution:
            print(f"           '{word}...': {count} ({count/len(data)*100:.0f}%)")
    
    # CHECK 5: System prompt consistency
    if format_type == "chatml":
        system_prompts = set()
        for item in data:
            sys_msgs = [m for m in item["messages"] if m["role"] == "system"]
            for m in sys_msgs:
                system_prompts.add(m["content"][:50])
        
        status = "PASS" if len(system_prompts) <= 2 else "WARN"
        print(f"  {'✅' if status=='PASS' else '⚠️'} Check 5 — System prompts: {len(system_prompts)} unique")
    
    # CHECK 6: Refusal contamination
    refusal_count = 0
    refusal_phrases = ["i cannot", "i can't", "i'm unable", "as an ai", "i don't have"]
    for item in data:
        if format_type == "chatml":
            for m in item["messages"]:
                if m["role"] == "assistant":
                    if any(p in m["content"].lower() for p in refusal_phrases):
                        refusal_count += 1
    
    refusal_pct = refusal_count / max(len(data), 1) * 100
    status = "PASS" if refusal_pct < 1 else "FAIL"
    print(f"  {'✅' if status=='PASS' else '❌'} Check 6 — Refusal contamination: {refusal_count} ({refusal_pct:.1f}%)")
    if refusal_count:
        print(f"           These teach the model to REFUSE. Remove them!")
    
    print(f"\n  SUMMARY: {stats}")
    return issues


# ---------- TEST WITH OUR DATA ----------

print("DATA QUALITY REPORT")
print("=" * 60)
print()

# Load the ChatML data we created
with open("training_data_chatml.jsonl") as f:
    data = [json.loads(line) for line in f]

issues = check_data_quality(data, "chatml")

# Now let's test with INTENTIONALLY BAD data
print("\n\n  --- Testing with BAD data ---\n")

bad_data = data.copy()
# Add a duplicate
bad_data.append(data[0])
# Add one with refusal
bad_data.append({
    "messages": [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "Tell me about refunds"},
        {"role": "assistant", "content": "I cannot help with that request."}
    ]
})
# Add one with empty response
bad_data.append({
    "messages": [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "Classify this"},
        {"role": "assistant", "content": ""}
    ]
})

issues = check_data_quality(bad_data, "chatml")

print("\n💡 Run quality checks BEFORE training. Finding issues after = wasted days.")
print("💡 Duplicates waste training budget. Refusals teach the model to refuse. Empty = errors.")

---# 3️⃣ Synthetic Data Generation (AI Creates Training Data)**Analogy:** You need 10,000 practice problems for a math textbook. Writing by hand takes months. Instead, you write 5 perfect example problems and ask a smart tutor (GPT-4) to create 10,000 more in the same style.**What we build:** Self-Instruct (generate from seeds) + Evol-Instruct (make questions progressively harder) + quality filtering.

In [ ]:
# ============================================================
# SYNTHETIC DATA: Use GPT-4 to generate training data
# Self-Instruct: generate from seed examples
# Evol-Instruct: make questions progressively harder
# ============================================================

# ---------- SEED EXAMPLES (you write 5 perfect ones) ----------

seeds = [
    {"instruction": "Classify sentiment", "input": "Great product, love it!", "output": "POSITIVE"},
    {"instruction": "Extract product and issue", "input": "My Dyson vacuum lost suction", "output": '{"product":"Dyson vacuum","issue":"lost suction"}'},
    {"instruction": "Summarize the complaint", "input": "Wrong size shipped, called 3 times, no resolution after 2 weeks", "output": "Wrong item shipped with no resolution despite multiple support calls over two weeks."},
    {"instruction": "Classify ticket priority", "input": "Database is completely down, all customers affected", "output": "CRITICAL"},
    {"instruction": "Write a polite response", "input": "This is terrible! I want my money back!", "output": "I sincerely apologize for your experience. Let me process your refund right away. Could you share your order number?"},
]


# ---------- SELF-INSTRUCT: Generate new examples from seeds ----------

def self_instruct(seeds, num_to_generate=5):
    """
    Show GPT-4 your seed examples.
    Ask it to generate NEW, DIFFERENT examples in the same style.
    """
    
    seed_text = json.dumps(seeds[:3], indent=2)
    
    generated = []
    
    for i in range(num_to_generate):
        prompt = (
            f"Here are example training data items:\n{seed_text}\n\n"
            f"Generate ONE new, completely DIFFERENT training example. "
            f"Use a different task type than the examples shown. "
            f"Return as JSON with keys: instruction, input, output. JSON only, no markdown."
        )
        
        result, tokens = ask(prompt, temperature=0.9, json_mode=True)
        try:
            item = json.loads(result)
            generated.append(item)
            print(f"  Generated {i+1}: {item.get('instruction','')[:50]}...")
        except:
            print(f"  Generated {i+1}: (failed to parse)")
    
    return generated


# ---------- EVOL-INSTRUCT: Make questions harder ----------

def evolve_difficulty(instruction, levels=3):
    """
    Take a simple question and make it progressively harder.
    Level 1: add specifics. Level 2: add constraints. Level 3: multi-step reasoning.
    """
    
    current = instruction
    evolution = [current]
    
    for level in range(levels):
        prompt = (
            f"Take this instruction and make it ONE level harder and more specific. "
            f"Add constraints, edge cases, or multi-step reasoning.\n\n"
            f"Current: {current}\n\n"
            f"Harder version (instruction only, no answer):"
        )
        current, _ = ask(prompt, temperature=0.7)
        evolution.append(current)
    
    return evolution


# ---------- QUALITY FILTER ----------

def filter_quality(items, min_score=7):
    """Use GPT-4 as a judge to score each generated example."""
    
    kept = []
    for item in items:
        score_text, _ = ask(
            f"Rate this training example 1-10 on quality (clear instruction, realistic input, correct output).\n"
            f"{json.dumps(item)}\n"
            f"Score (number only):"
        )
        try:
            score = int(score_text.strip().split()[0])
        except:
            score = 5
        
        status = "KEEP" if score >= min_score else "TOSS"
        print(f"  [{status}] Score {score}/10: {item.get('instruction','')[:40]}...")
        if score >= min_score:
            kept.append(item)
    
    return kept


# ---------- RUN ----------

print("SELF-INSTRUCT (generate from seeds)")
print("=" * 60)
generated = self_instruct(seeds, num_to_generate=5)

print(f"\n\nEVOL-INSTRUCT (progressive difficulty)")
print("=" * 60)
evolution = evolve_difficulty("Classify the sentiment of this review")
for i, level in enumerate(evolution):
    print(f"  Level {i}: {level[:70]}...")

print(f"\n\nQUALITY FILTER (keep only good examples)")
print("=" * 60)
filtered = filter_quality(generated, min_score=7)
print(f"\n  Generated: {len(generated)} | Kept: {len(filtered)} | Filtered out: {len(generated)-len(filtered)}")

print(f"\n💡 Self-Instruct: 5 seeds -> unlimited new examples. GPT-4 does the work.")
print(f"💡 Evol-Instruct: simple questions become progressively harder. Smarter models.")
print(f"💡 Quality filter: generate 2x what you need, keep the top 50%.")
print(f"💡 Cost: 1000 examples via GPT-4 = ~$5-10. Manual writing = $2000+.")

---# 4️⃣ Fine-Tune via OpenAI API (No GPU Needed!)**Analogy:** Ordering from a restaurant vs cooking at home. You give them the recipe (your data), they cook it (fine-tune on their servers), and you get a custom dish back (your custom model ID). No kitchen required.**What we do:** Upload training data → start fine-tuning → get custom model ID. All via API.

In [ ]:
# ============================================================
# OPENAI FINE-TUNING: Upload data, get a custom model back
# No GPU. No infrastructure. Just API calls.
# ============================================================

# ---------- STEP 1: Prepare the data file ----------

# We need at least 10 examples for OpenAI (50+ recommended)
# Let's create a realistic small dataset

training_examples = [
    ("Classify sentiment: 'Best phone I ever owned!'", "POSITIVE"),
    ("Classify sentiment: 'Battery dies in 2 hours. Terrible.'", "NEGATIVE"),
    ("Classify sentiment: 'It works fine, nothing special.'", "NEUTRAL"),
    ("Classify sentiment: 'Absolutely love the camera quality!'", "POSITIVE"),
    ("Classify sentiment: 'Broke after one week. Waste of money.'", "NEGATIVE"),
    ("Classify sentiment: 'Decent product for the price.'", "NEUTRAL"),
    ("Classify sentiment: 'Customer service was incredibly helpful!'", "POSITIVE"),
    ("Classify sentiment: 'Arrived damaged, very disappointed.'", "NEGATIVE"),
    ("Classify sentiment: 'Does what it says. No complaints.'", "NEUTRAL"),
    ("Classify sentiment: 'My kids love this toy! Great purchase.'", "POSITIVE"),
]

# Format for OpenAI
openai_training_data = []
for user_msg, assistant_msg in training_examples:
    openai_training_data.append({
        "messages": [
            {"role": "system", "content": "You are a sentiment classifier. Respond with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL."},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
    })

# Save as JSONL (one JSON per line — required format)
with open("openai_finetune_data.jsonl", "w") as f:
    for item in openai_training_data:
        f.write(json.dumps(item) + "\n")

print("OPENAI FINE-TUNING")
print("=" * 60)
print(f"\n  Step 1: Created {len(openai_training_data)} training examples")
print(f"  Saved to: openai_finetune_data.jsonl")


# ---------- STEP 2: Upload the file ----------

print("\n  Step 2: Uploading file to OpenAI...")
try:
    upload = client.files.create(
        file=open("openai_finetune_data.jsonl", "rb"),
        purpose="fine-tune"
    )
    print(f"  ✅ Uploaded! File ID: {upload.id}")
    
    # ---------- STEP 3: Start fine-tuning ----------
    
    print("\n  Step 3: Starting fine-tuning job...")
    print("  (This would take 10-30 minutes for a small dataset)")
    print()
    print("  To actually start the job, uncomment this code:")
    print("  ─" * 30)
    print("""
    # job = client.fine_tuning.jobs.create(
    #     training_file=upload.id,
    #     model="gpt-4o-mini-2024-07-18",   # Base model to fine-tune
    #     hyperparameters={
    #         "n_epochs": 3,                 # How many times to go through data
    #     }
    # )
    # print(f"Job started: {job.id}")
    # print(f"Status: {job.status}")
    #
    # # Check status:
    # # status = client.fine_tuning.jobs.retrieve(job.id)
    # # print(status.status)  # "running" -> "succeeded"
    # # print(status.fine_tuned_model)  # "ft:gpt-4o-mini-2024-07-18:org:name:id"
    #
    # # Use your custom model:
    # # response = client.chat.completions.create(
    # #     model="ft:gpt-4o-mini-2024-07-18:your-org:your-name:abc123",
    # #     messages=[{"role":"user","content":"Classify: 'Amazing product!'"}]
    # # )
    """)
    
except Exception as e:
    print(f"  ⚠️ Upload skipped: {e}")
    print("  (Set OPENAI_API_KEY to upload)")

print("\n  COST ESTIMATE:")
print("  ─" * 30)
tokens_per_example = 50  # Average for our data
total_tokens = len(openai_training_data) * tokens_per_example * 3  # 3 epochs
cost = total_tokens / 1_000_000 * 3.0  # gpt-4o-mini fine-tuning price
print(f"  {len(openai_training_data)} examples x 3 epochs = ~{total_tokens:,} training tokens")
print(f"  Estimated cost: ${cost:.4f}")
print(f"  Time: ~10-30 minutes")
print(f"\n💡 No GPU. No infrastructure. Upload data, get a custom model.")
print(f"💡 50+ examples recommended. 1000+ for best results.")
print(f"💡 Fine-tuned gpt-4o-mini often beats base gpt-4o on YOUR specific task.")

---# 5️⃣ DPO Preference Data (Teach the Model Which Answer is Better)**Analogy:** Two restaurant dishes that are both correct — pasta with red sauce and pasta with white sauce. Alignment is like taste-testing with customers and learning that 70% prefer the red sauce. DPO trains the model to prefer what humans prefer.**What we build:** Generate preference pairs (chosen vs rejected) + use GPT-4 as a judge (RLAIF).

In [ ]:
# ============================================================
# DPO: Create preference pairs (good response vs bad response)
# The model learns: "make outputs more like the good one"
# ============================================================

# ---------- METHOD 1: Manual preference pairs ----------

manual_pairs = [
    {
        "prompt": "Explain what machine learning is.",
        "chosen": "Machine learning is a way for computers to learn from examples instead of being explicitly programmed. For instance, instead of writing rules to detect spam emails, you show the computer thousands of spam and non-spam emails, and it figures out the patterns itself.",
        "rejected": "Machine learning is a very complex field involving statistical methods, optimization algorithms, and various mathematical frameworks that enable computational systems to improve their performance on tasks through experience, leveraging techniques such as gradient descent and backpropagation in neural network architectures."
    },
    {
        "prompt": "How do I reset my password?",
        "chosen": "Go to Settings > Security > Change Password. Enter your current password, then your new one twice. Click Save.",
        "rejected": "I'd be happy to help you with that! Password resets can sometimes be tricky, but don't worry! There are several ways you might be able to reset your password depending on your account type and security settings. Let me walk you through some general steps that might help..."
    },
]

print("DPO PREFERENCE DATA")
print("=" * 60)
print()
print("METHOD 1 — Manual pairs (highest quality, most expensive):")
for pair in manual_pairs:
    print(f"\n  Prompt: {pair['prompt'][:50]}...")
    print(f"  ✅ Chosen:  {pair['chosen'][:60]}...")
    print(f"  ❌ Rejected: {pair['rejected'][:60]}...")


# ---------- METHOD 2: RLAIF (GPT-4 judges instead of humans) ----------

def generate_preference_pair(prompt):
    """
    RLAIF: use GPT-4 as a judge.
    1. Generate 2 different responses
    2. Ask GPT-4 which is better
    3. Return as a preference pair
    """
    
    # Generate response A (creative)
    response_a, _ = ask(prompt, temperature=0.9)
    
    # Generate response B (different approach)
    response_b, _ = ask(
        f"Answer this differently than: '{response_a[:50]}...'\n\nQuestion: {prompt}",
        temperature=0.9
    )
    
    # GPT-4 judges which is better
    judgment, _ = ask(
        f"Which response is better? Consider: accuracy, clarity, conciseness.\n\n"
        f"Question: {prompt}\n\n"
        f"Response A: {response_a}\n\n"
        f"Response B: {response_b}\n\n"
        f"Reply with ONLY 'A' or 'B'."
    )
    
    if "A" in judgment.upper():
        return {"prompt": prompt, "chosen": response_a, "rejected": response_b, "judge": "A"}
    else:
        return {"prompt": prompt, "chosen": response_b, "rejected": response_a, "judge": "B"}


print("\n\nMETHOD 2 — RLAIF (GPT-4 judges, 80% quality at 10% cost):")
print("─" * 50)

test_prompts = [
    "What is the refund policy for electronics?",
    "Explain cloud computing to a 10-year-old.",
    "How do I improve my Python code performance?",
]

dpo_dataset = []
for prompt in test_prompts:
    pair = generate_preference_pair(prompt)
    dpo_dataset.append(pair)
    print(f"\n  Q: {prompt[:50]}...")
    print(f"  Judge picked: Response {pair['judge']}")
    print(f"  ✅ Chosen:  {pair['chosen'][:60]}...")
    print(f"  ❌ Rejected: {pair['rejected'][:60]}...")

# Save
with open("dpo_training_data.jsonl", "w") as f:
    for pair in dpo_dataset:
        f.write(json.dumps(pair) + "\n")

print(f"\n\n  Saved {len(dpo_dataset)} preference pairs to dpo_training_data.jsonl")
print(f"\n💡 Manual pairs: highest quality. Use for 20% of your dataset.")
print(f"💡 RLAIF (GPT-4 judge): 80% as good, 10% of the cost. Use for the other 80%.")
print(f"💡 Pipeline: SFT first (instruction following) -> THEN DPO (alignment).")
print(f"💡 Never skip SFT. DPO on a model that can't follow instructions = useless.")

---# 6️⃣ LoRA / QLoRA Config (Copy-Paste for Google Colab)**Analogy:** Customizing a Boeing 747. You don't rebuild the whole plane. You ADD a first-class cabin alongside the existing structure. LoRA adds small trainable layers alongside frozen model weights. Only 0.5% of the model is trained. The rest stays frozen.**This code:** Runs on Google Colab (free T4 GPU). Copy, paste, run.

In [ ]:
# ============================================================
# LoRA / QLoRA: The config explained line by line
# Copy this to Google Colab (free T4 GPU) to actually run it
# ============================================================

print("LoRA / QLoRA CONFIGURATION (explained)")
print("=" * 60)
print()

# This is the EXACT code you paste into Google Colab:
colab_code = '''
# Step 1: Install (run in Colab)
# !pip install unsloth transformers trl datasets

# Step 2: Load model in 4-bit (QLoRA = quantized LoRA)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",  # 8B model compressed to 4-bit
    max_seq_length=2048,                         # How many words per example
    load_in_4bit=True,                           # QLoRA: compress to save memory
)

# Step 3: Add LoRA adapters (the small trainable part)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                        # Rank: 16 = good default
    target_modules=["q_proj","k_proj","v_proj","o_proj"],  # Which layers to adapt
    lora_alpha=16,                               # Scaling factor (usually = r)
    lora_dropout=0,                              # Dropout (0 = no regularization)
    bias="none",                                 # Don't train bias terms
)

# Step 4: Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Load your data (upload to Colab first)
dataset = load_dataset("json", data_files="training_data_chatml.jsonl")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    args=TrainingArguments(
        output_dir="./lora_output",
        per_device_train_batch_size=2,           # How many examples at once
        gradient_accumulation_steps=4,           # Simulate bigger batch
        num_train_epochs=3,                      # 3 passes through data
        learning_rate=2e-4,                      # LoRA uses higher LR than full FT
        fp16=True,                               # Half precision (faster)
        logging_steps=10,
        save_strategy="epoch",
    ),
)

trainer.train()  # This takes 10-60 minutes depending on data size

# Step 5: Save and merge
model.save_pretrained_merged("./merged_model", tokenizer)
# This merged model runs like a normal model — zero LoRA overhead
'''

# Print with explanations
for line in colab_code.strip().split("\n"):
    if line.strip().startswith("#"):
        print(f"  {line}")
    elif line.strip():
        print(f"  {line}")
    else:
        print()

print()
print("KEY NUMBERS:")
print("  ─" * 30)

total_params = 8_030_000_000
lora_params = 41_943_040
frozen_params = total_params - lora_params

print(f"  Total parameters:    {total_params:>15,}")
print(f"  Trainable (LoRA):    {lora_params:>15,}  ({lora_params/total_params*100:.2f}%)")
print(f"  Frozen:              {frozen_params:>15,}  ({frozen_params/total_params*100:.2f}%)")
print()
print("  Memory: 70B full = 140GB (4 GPUs). 70B QLoRA = 35GB (1 GPU!).")
print("  Cost:   70B full fine-tune ~$640. 70B QLoRA ~$8.")
print()
print("💡 QLoRA is the default. Train 0.5% of the model. Get 95-99% of full quality.")
print("💡 Unsloth: 2x faster, 60% less memory. Always use it.")
print("💡 After training: MERGE adapters into base model. Zero inference overhead.")

---# 7️⃣ Cost Calculator — Fine-Tune vs RAG vs Prompt Engineering**The big question:** Should you fine-tune, use RAG, or just use better prompts? This calculator shows the REAL costs so you can make an informed decision.

In [ ]:
# ============================================================
# COST COMPARISON: All approaches side by side
# ============================================================

print("FINE-TUNING DECISION CALCULATOR")
print("=" * 60)

# Scenario: 100K queries per day for a classification task

queries_per_day = 100_000
days_per_month = 30
avg_tokens_per_query = 200

print(f"\n  Scenario: {queries_per_day:,} queries/day, {avg_tokens_per_query} tokens each\n")

approaches = [
    {
        "name": "GPT-4o (no fine-tuning)",
        "per_token_cost": 5.0 / 1_000_000,   # $5 per 1M tokens
        "setup_cost": 0,
        "setup_time": "0 days",
        "accuracy": "90%",
        "note": "Best quality, most expensive per call"
    },
    {
        "name": "GPT-4o-mini (no fine-tuning)",
        "per_token_cost": 0.15 / 1_000_000,
        "setup_cost": 0,
        "setup_time": "0 days",
        "accuracy": "85%",
        "note": "Good quality, very cheap"
    },
    {
        "name": "GPT-4o-mini (fine-tuned)",
        "per_token_cost": 0.30 / 1_000_000,   # Fine-tuned costs 2x
        "setup_cost": 25,                       # ~$25 for training
        "setup_time": "2-3 days",
        "accuracy": "93%",
        "note": "Fine-tuned mini often beats base GPT-4o on YOUR task"
    },
    {
        "name": "Self-hosted Llama-8B (QLoRA fine-tuned)",
        "per_token_cost": 0.02 / 1_000_000,   # Just GPU electricity
        "setup_cost": 500,                      # GPU time for training + setup
        "setup_time": "5-7 days",
        "accuracy": "91%",
        "note": "Cheapest at scale, needs GPU infrastructure"
    },
    {
        "name": "GPT-4o-mini + RAG (no fine-tuning)",
        "per_token_cost": 0.15 / 1_000_000 * 3,  # 3x tokens (context stuffing)
        "setup_cost": 100,                     # Embedding + vector DB setup
        "setup_time": "3-5 days",
        "accuracy": "88%",
        "note": "Best for dynamic knowledge that changes often"
    },
]

print(f"{'Approach':<40} {'Monthly':>10} {'Setup':>8} {'Time':>8} {'Accuracy':>9}")
print("─" * 80)

for a in approaches:
    monthly_tokens = queries_per_day * avg_tokens_per_query * days_per_month
    monthly_cost = monthly_tokens * a["per_token_cost"]
    
    print(f"{a['name']:<40} ${monthly_cost:>8,.0f} ${a['setup_cost']:>6,.0f} {a['setup_time']:>8} {a['accuracy']:>9}")

print()
print("DECISION GUIDE:")
print("  ─" * 30)
print("  Just need it to work NOW? → GPT-4o-mini (zero setup)")
print("  Need max accuracy on YOUR task? → Fine-tune GPT-4o-mini ($25 setup)")
print("  High volume (100K+/day)? → Self-host fine-tuned Llama (cheapest)")
print("  Knowledge changes weekly? → RAG (no retraining needed)")
print("  Best combo: Fine-tune for FORMAT + RAG for KNOWLEDGE")
print()
print("💡 Fine-tuned GPT-4o-mini often beats base GPT-4o on specific tasks.")
print("💡 Fine-tuning teaches the model HOW to respond. RAG gives it WHAT to know.")
print("💡 They complement each other. Not either/or.")

---# 🏁 Summary — Fine-Tuning Decision Tree```Do you need to change the model's BEHAVIOR (tone, format, style)?  YES → Fine-tune    Have GPU? → QLoRA with Unsloth (cheapest)    No GPU?   → OpenAI fine-tuning API (easiest)  NO → Don't fine-tune. Use prompts or RAG.Do you need to give the model NEW KNOWLEDGE?  Knowledge changes often?  → RAG (retrieve at query time)  Knowledge is static?      → Continued pre-training OR RAG  Need DEEP domain understanding? → Continued pre-training + fine-tunePipeline: Continued Pre-Training (knowledge) → SFT (skills) → DPO (alignment)```**Golden rules:**1. Try prompts first. Fine-tune only when prompts aren't enough.2. QLoRA is the default. Full fine-tuning only if QLoRA quality gap > 2%.3. 1000 curated examples > 100,000 sloppy ones (LIMA paper).4. Every production bug → add to training data. Your dataset improves from experience.